# 消息 DAG：按样本和目标查看

图不使用标签构建。红=幻觉，绿=正常，灰=特殊。可切换来源和物理 head；显示覆盖率说明稀疏边保留了多少贡献。

In [ ]:
from pathlib import Path
import json, sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'experiments/reanchor_flow/message_dag').is_dir())
sys.path.insert(0, str(PROJECT))
OUTPUT = Path.cwd() if (Path.cwd()/'index.json').is_file() else PROJECT/'experiments/reanchor_flow/outputs/attention_audit_v3/message_dag_v2'
index = json.loads((OUTPUT/'index.json').read_text())
[(e['split'], e['task_type'], e['sample_id'], e['targets']) for e in index['samples']]

In [ ]:
available = [e for e in index['samples'] if any((OUTPUT/e['folder']/f'target_{t}.npz').exists() for t in e['targets'])]
if not available: raise ValueError('No completed targets in this output directory')
SPLIT, TASK, SAMPLE_ID = available[0]['split'], available[0]['task_type'], available[0]['sample_id']  # 可改为上方实际编号
rows = [e for e in available if e['split']==SPLIT and e['task_type']==TASK and str(e['sample_id'])==str(SAMPLE_ID)]
if not rows: raise ValueError('Requested sample is not in the available list above')
e = rows[0]
folder = OUTPUT/e['folder']
ready = [t for t in e['targets'] if (folder/f'target_{t}.npz').exists()]
print('sample:', e['sample_id'], 'completed targets:', ready)


In [ ]:
TARGET = next((d['position'] for d in e.get('target_details', []) if d['position'] in ready and d['content_candidate']), ready[0])  # 可改为任意已完成目标
from experiments.reanchor_flow.message_dag.view import render_target
from IPython.display import IFrame, display
import base64
page = render_target(folder, TARGET)
url = 'data:text/html;base64,' + base64.b64encode(page.read_bytes()).decode()
display(IFrame(url, width='100%', height=1100))

In [ ]:
# 整体逐 layer/head 的 N/H 统计，而不是只看一个样本
from IPython.display import Image
figure = OUTPUT/'cohorts'/f'{SPLIT}_{TASK}_heads.png'
display(Image(filename=str(figure)))